In [0]:
with hcp_hco_master as
(select 
a.npi_num__v as hcp_npi,
a.vid__v     as hcp_vid,
c.npi_num__v as hco_npi,
b.parent_hco_vid__v as hco_vid,
c.corporate_name__v as hco_name,
b.modified_date__v as modified_date,
b.status_update_time__v as status_update_time,
ROW_NUMBER() OVER (
      PARTITION BY A.vid__v
      ORDER BY B.modified_date__v DESC NULLS LAST, 
               B.status_update_time__v DESC NULLS LAST
    ) AS rn
from com_edp_prd.com_raw.vod_hcp a
left join com_edp_prd.com_raw.vod_parenthco b 
on a.vid__v = b.entity_vid__v and b.hierarchy_type__v = 'HCP_HCO' and B.PARENT_HCO_STATUS__V = 'A' and b.relationship_type__v = '7356'
left join com_edp_prd.com_raw.vod_hco c 
on b.parent_hco_vid__v = c.vid__v
),
final_hcp_hco_unique_affiliation as (
  select 
hcp_npi,
hcp_vid,
hco_npi,
hco_vid,
hco_name
 from hcp_hco_master
where rn = 1
),
hcp_hco_affiliation_demographic_info as (
  select 
a.hcp_vid,
a.hcp_npi,
concat(b.first_name,' ',b.last_name) as hcp_name,
b.primary_specialty as hcp_primary_specialty,
a.hco_vid,
a.hco_npi,
a.hco_name
from final_hcp_hco_unique_affiliation a
left join com_edp_prd.com_raw.kom_providers b
on a.hcp_npi = b.npi and b.provider_type = 'INDIVIDUAL'
where a.hcp_npi is not null and a.hcp_npi not in ('1063476125', '1912920919', '-')),

hcp_hco_affiliation_with_hco_zip as (
  select 
  a.*,
  b.postal_code_cda__v as hco_zip,
  b.modified_date__v,
  row_number() over (partition by a.hco_vid order by b.modified_date__v desc) as rn
  from hcp_hco_affiliation_demographic_info a
  left join com_edp_prd.com_raw.vod_address b
  on b.entity_vid__v = a.hco_vid and b.entity_type__v = 'HCO'
   AND b.record_state__v = 'VALID'
   AND b.address_status__v IN ('A','DS')
   AND b.address_verification_status__v NOT IN ('NS', 'U')
)
select
hcp_vid,
  hcp_npi,
  hcp_name,
  hcp_primary_specialty,
  hco_vid,
  hco_npi,
  hco_name,
  hco_zip
FROM hcp_hco_affiliation_with_hco_zip
where rn = 1
ORDER BY hcp_npi;


In [0]:
create or replace table com_edp_prd.cmpa_insights_internal_schema.hcp_hco_affiliations_master_table
as (with hcp_hco_master as
(select 
a.npi_num__v as hcp_npi,
a.vid__v     as hcp_vid,
c.npi_num__v as hco_npi,
b.parent_hco_vid__v as hco_vid,
c.corporate_name__v as hco_name,
b.modified_date__v as modified_date,
b.status_update_time__v as status_update_time,
ROW_NUMBER() OVER (
      PARTITION BY A.vid__v
      ORDER BY B.modified_date__v DESC NULLS LAST, 
               B.status_update_time__v DESC NULLS LAST
    ) AS rn
from com_edp_prd.com_raw.vod_hcp a
left join com_edp_prd.com_raw.vod_parenthco b 
on a.vid__v = b.entity_vid__v and b.hierarchy_type__v = 'HCP_HCO' and B.PARENT_HCO_STATUS__V = 'A' and b.relationship_type__v = '7356'
left join com_edp_prd.com_raw.vod_hco c 
on b.parent_hco_vid__v = c.vid__v
),
final_hcp_hco_unique_affiliation as (
  select 
hcp_npi,
hcp_vid,
hco_npi,
hco_vid,
hco_name
 from hcp_hco_master
where rn = 1
),
hcp_hco_affiliation_demographic_info as (
  select 
a.hcp_vid,
a.hcp_npi,
concat(b.first_name,' ',b.last_name) as hcp_name,
b.primary_specialty as hcp_primary_specialty,
a.hco_vid,
a.hco_npi,
a.hco_name
from final_hcp_hco_unique_affiliation a
left join com_edp_prd.com_raw.kom_providers b
on a.hcp_npi = b.npi and b.provider_type = 'INDIVIDUAL'
where a.hcp_npi is not null and a.hcp_npi not in ('1063476125', '1912920919', '-')),

hcp_hco_affiliation_with_hco_zip as (
  select 
  a.*,
  b.postal_code_cda__v as hco_zip,
  b.modified_date__v,
  row_number() over (partition by a.hco_vid order by b.modified_date__v desc) as rn
  from hcp_hco_affiliation_demographic_info a
  left join com_edp_prd.com_raw.vod_address b
  on b.entity_vid__v = a.hco_vid and b.entity_type__v = 'HCO'
   AND b.record_state__v = 'VALID'
   AND b.address_status__v IN ('A','DS')
   AND b.address_verification_status__v NOT IN ('NS', 'U')
)
select
hcp_vid,
  hcp_npi,
  hcp_name,
  hcp_primary_specialty,
  hco_vid,
  hco_npi,
  hco_name,
  hco_zip
FROM hcp_hco_affiliation_with_hco_zip
where rn = 1
ORDER BY hcp_npi
)

In [0]:
select * from com_edp_prd.cmpa_insights_internal_schema.hcp_hco_affiliations_master_table

### Adding Specialty in the reference table

In [0]:
select * from com_edp_prd.cmpa_insights_internal_schema.hcp_hco_affiliations_master_table

In [0]:

create or replace table com_edp_prd.cmpa_insights_internal_schema.hcp_hco_affiliations_master_table
select a.`HCP NPI` as hcp_npi, a.`HCP First Name` as hcp_first_name, a.`HCP Last Name` as hcp_last_name, a.`HCP Name` as hcp_name, b.PRIMARY_SPECIALTY as hcp_specialty, a.`HCP Zipcode` as hcp_zip, a.`HCO 
Primary NPI` as hco_npi, a.`HCO Primary Name` as hco_name, a.`HCO Zip` as hco_zip, a.`Final Zip` as final_zip, a.`Mapped Territory` as territory, a.Region as region 
from com_edp_prd.cmpa_insights_internal_schema.hcp_hco_affiliations_master_table as a
left join com_raw.kom_providers as b on a.`HCP NPI` = b.NPI and b.PROVIDER_TYPE = 'INDIVIDUAL'

In [0]:
select * from com_edp_prd.cmpa_insights_internal_schema.hcp_hco_affiliations_master_table

In [0]:
select * from com_edp_prd.cmpa_insights_internal_schema.mpsii_elaprase_patient_info